# OASIS-3: robustness, negative controls, horizon analysis, and external-transfer checks

**Standalone continuation notebook.**

This notebook does not depend on the original modeling notebook being loaded in the kernel. It loads the saved outer-test prediction artifacts from disk and performs robustness analyses. Where a test genuinely requires a different representation or a new prediction task, the notebook provides an explicit, self-contained section or a clear checkpoint rather than assuming in-memory variables exist.

In [1]:
from pathlib import Path
import gc, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATASET_ROOT = Path(r"C:\Projects\OASIS\oasis_raw_t1_encoder")
PAPER_ROOT = Path(r"C:\Projects\OASIS\oasis_raw_t1_encoder/PAPER_READY_FINAL_RESULTS")
REPEAT_ROOT = PAPER_ROOT / "outer_repeats"
print("Dataset root:", DATASET_ROOT)
print("Paper root:", PAPER_ROOT)
if not PAPER_ROOT.exists():
    raise FileNotFoundError(f"Paper output directory not found: {PAPER_ROOT}")
if not REPEAT_ROOT.exists():
    raise FileNotFoundError(f"Outer-repeat directory not found: {REPEAT_ROOT}")


Dataset root: C:\Projects\OASIS\oasis_raw_t1_encoder
Paper root: C:\Projects\OASIS\oasis_raw_t1_encoder\PAPER_READY_FINAL_RESULTS


## 1. Load saved outer-repeat artifacts

In [2]:
def load_all_repeats():
    test_parts, oof_parts = [], []
    for rep_dir in sorted(REPEAT_ROOT.glob("repeat_*")):
        rep = int(rep_dir.name.split("_")[-1])
        tf = rep_dir / "test.csv"
        of = rep_dir / "oof.csv"
        if not tf.exists() or not of.exists():
            raise FileNotFoundError(f"Missing test.csv or oof.csv in {rep_dir}")
        t = pd.read_csv(tf); o = pd.read_csv(of)
        t["repeat"] = rep; o["repeat"] = rep
        test_parts.append(t); oof_parts.append(o)
    if not test_parts:
        raise FileNotFoundError(f"No completed outer repeats found in {REPEAT_ROOT}")
    return pd.concat(test_parts, ignore_index=True), pd.concat(oof_parts, ignore_index=True)

test_all, oof_all = load_all_repeats()
print("Repeats:", sorted(test_all.repeat.unique().tolist()))
print("Held-out rows:", len(test_all), "OOF rows:", len(oof_all))
for rep in sorted(test_all.repeat.unique()):
    assert set(test_all.loc[test_all.repeat.eq(rep),"subject_id"]).isdisjoint(
        set(oof_all.loc[oof_all.repeat.eq(rep),"subject_id"])
    )
print("PASS: subject-disjoint OOF/test separation is preserved.")


Repeats: [1, 2, 3, 4, 5, 6]
Held-out rows: 311 OOF rows: 1219
PASS: subject-disjoint OOF/test separation is preserved.


## 2. Negative-control experiment: shuffled cross-modal pairing

In [3]:
def shuffled_pairing_null(test_df, permutations=1000, seed=42):
    rng = np.random.default_rng(seed)
    real_rows, null_rows = [], []
    for rep, g0 in test_df.groupby("repeat"):
        g = g0.reset_index(drop=True).copy()
        real_joint = (g.raw_prediction + g.clinical_prediction) / 2.0
        real_score = (g.raw_prediction - g.clinical_prediction).abs()
        real_err = (real_joint - g.future_mmse).abs()
        real = pd.DataFrame({"subject_id":g.subject_id,"score":real_score,"err":real_err})
        real = real.groupby("subject_id",as_index=False).mean(numeric_only=True)
        rho = spearmanr(real.score, real.err).statistic if len(real)>=10 and real.score.nunique()>1 and real.err.nunique()>1 else np.nan
        real_rows.append({"repeat":rep,"real_rho":rho,"N_subjects":len(real)})
        for p in range(permutations):
            idx = rng.permutation(len(g))
            pseudo_joint = (g.raw_prediction.to_numpy()[idx] + g.clinical_prediction.to_numpy()) / 2.0
            pseudo_score = np.abs(g.raw_prediction.to_numpy()[idx] - g.clinical_prediction.to_numpy())
            pseudo_err = np.abs(pseudo_joint - g.future_mmse.to_numpy())
            d = pd.DataFrame({"subject_id":g.subject_id,"score":pseudo_score,"err":pseudo_err})
            d = d.groupby("subject_id",as_index=False).mean(numeric_only=True)
            rr = spearmanr(d.score,d.err).statistic if len(d)>=10 and d.score.nunique()>1 and d.err.nunique()>1 else np.nan
            null_rows.append({"repeat":rep,"permutation":p,"rho":rr})
    return pd.DataFrame(real_rows), pd.DataFrame(null_rows)

real_nc, null_nc = shuffled_pairing_null(test_all, permutations=1000)
summary_nc=[]
for rep in real_nc.repeat:
    rr=float(real_nc.loc[real_nc.repeat.eq(rep),"real_rho"].iloc[0])
    null=null_nc.loc[null_nc.repeat.eq(rep),"rho"].dropna().to_numpy()
    summary_nc.append({
        "repeat":rep, "real_rho":rr,
        "null_mean":np.mean(null) if len(null) else np.nan,
        "null_sd":np.std(null,ddof=1) if len(null)>1 else np.nan,
        "empirical_p_one_sided":np.mean(null>=rr) if len(null) else np.nan,
        "null_N":len(null)
    })
negative_control=pd.DataFrame(summary_nc)
display(negative_control)


,repeat,real_rho,null_mean,null_sd,empirical_p_one_sided,null_N
0,1,0.309272,0.559186,0.095067,0.988,1000
1,2,0.339928,0.526357,0.089723,0.979,1000
2,3,0.413168,0.507273,0.093628,0.840,1000
3,4,0.277107,0.379462,0.103698,0.825,1000
4,5,0.495923,0.373372,0.110003,0.126,1000
5,6,0.415433,0.479101,0.095917,0.751,1000


## 3. Forecast-horizon analysis

In [4]:
if "future_interval_years" not in test_all.columns and "future_interval_days" in test_all.columns:
    test_all["future_interval_years"] = test_all["future_interval_days"] / 365.25
bins=[0.5,1.0,1.5,2.0,99]
labels=["0.5-1.0 y","1.0-1.5 y","1.5-2.0 y","2.0+ y"]
test_all["horizon_bin"]=pd.cut(test_all["future_interval_years"],bins=bins,labels=labels,right=False)
rows=[]
for (rep,h),g in test_all.groupby(["repeat","horizon_bin"],observed=True):
    d=g[["subject_id","prediction_disagreement","uncertainty","abs_error"]].dropna()
    d=d.groupby("subject_id",as_index=False).mean(numeric_only=True)
    if len(d)<10: continue
    for score in ["prediction_disagreement","uncertainty"]:
        rho,p=(np.nan,np.nan)
        if d[score].nunique()>1 and d.abs_error.nunique()>1:
            rho,p=spearmanr(d[score],d.abs_error)
        rows.append({"repeat":rep,"horizon":str(h),"score":score,"N_subjects":len(d),"rho":rho,"p":p})
horizon_df=pd.DataFrame(rows)
display(horizon_df)


,repeat,horizon,score,N_subjects,rho,p
0,1,0.5-1.0 y,prediction_disagreement,19,-0.138596,0.571491
1,1,0.5-1.0 y,uncertainty,19,0.363158,0.126455
2,1,1.0-1.5 y,prediction_disagreement,26,0.209573,0.304172
3,1,1.0-1.5 y,uncertainty,26,0.197265,0.334082
4,2,0.5-1.0 y,prediction_disagreement,24,0.306957,0.144568
5,2,0.5-1.0 y,uncertainty,24,-0.145217,0.498379
6,2,1.0-1.5 y,prediction_disagreement,23,0.442688,0.034398
7,2,1.0-1.5 y,uncertainty,23,0.449605,0.031363
8,3,0.5-1.0 y,prediction_disagreement,15,0.153571,0.584764
9,3,0.5-1.0 y,uncertainty,15,0.335714,0.221212


## 4. History-availability and baseline-difficulty sensitivity

In [5]:
rows=[]
for rep,g in test_all.groupby("repeat"):
    for variable in ["n_mri","n_clinical","anchor_mmse"]:
        if variable not in g.columns: continue
        cut=float(g[variable].median())
        for level,gg in [("low",g[g[variable]<=cut]),("high",g[g[variable]>cut])]:
            d=gg[["subject_id","prediction_disagreement","uncertainty","abs_error"]].dropna()
            d=d.groupby("subject_id",as_index=False).mean(numeric_only=True)
            if len(d)<10: continue
            for score in ["prediction_disagreement","uncertainty"]:
                rho=spearmanr(d[score],d.abs_error).statistic if d[score].nunique()>1 and d.abs_error.nunique()>1 else np.nan
                rows.append({"repeat":rep,"stratifier":variable,"level":level,"score":score,"N_subjects":len(d),"rho":rho})
sensitivity_df=pd.DataFrame(rows)
display(sensitivity_df)


,repeat,stratifier,level,score,N_subjects,rho
0,1,n_mri,low,prediction_disagreement,36,0.238095
1,1,n_mri,low,uncertainty,36,0.240927
2,1,n_mri,high,prediction_disagreement,12,0.132867
3,1,n_mri,high,uncertainty,12,0.258741
4,1,n_clinical,low,prediction_disagreement,25,0.202308
...,...,...,...,...,...,...
65,6,n_clinical,high,uncertainty,20,-0.010526
66,6,anchor_mmse,low,prediction_disagreement,28,0.390257
67,6,anchor_mmse,low,uncertainty,28,0.234264
68,6,anchor_mmse,high,prediction_disagreement,17,0.617647


## 5. Modality corruption / branch ablation on held-out predictions

In [6]:
# Prediction-level corruption, using no outcomes to define the corruption.
# This does not replace the raw MRI experiment; it is a cheap falsification/sensitivity control.
rows=[]
for rep,g in test_all.groupby("repeat"):
    g=g.copy()
    y=g.future_mmse.to_numpy()
    for mode in ["real","shuffle_mri","shuffle_clinical","clinical_only","mri_only"]:
        rng=np.random.default_rng(SEED+1000+rep)
        rp=g.raw_prediction.to_numpy().copy()
        cp=g.clinical_prediction.to_numpy().copy()
        if mode=="shuffle_mri": rp=rng.permutation(rp)
        if mode=="shuffle_clinical": cp=rng.permutation(cp)
        if mode=="clinical_only": pred=cp
        elif mode=="mri_only": pred=rp
        else: pred=(rp+cp)/2.0
        err=np.abs(pred-y)
        score=np.abs(rp-cp)
        rows.append({"repeat":rep,"mode":mode,
                     "MAE":float(np.mean(err)),
                     "disagreement_error_rho":float(spearmanr(score,err).statistic) if np.unique(score).size>1 and np.unique(err).size>1 else np.nan})
corruption_df=pd.DataFrame(rows)
display(corruption_df)


,repeat,mode,MAE,disagreement_error_rho
0,1,real,1.315241,0.333562
1,1,shuffle_mri,1.404511,0.526434
2,1,shuffle_clinical,1.731847,0.475967
3,1,clinical_only,1.055421,-0.016810
4,1,mri_only,1.709566,0.437621
5,2,real,1.581045,0.338441
6,2,shuffle_mri,1.764771,0.727921
7,2,shuffle_clinical,2.007908,0.275777
8,2,clinical_only,1.370150,0.202592
9,2,mri_only,2.022505,0.567753


## 6. FreeSurfer benchmark checkpoint
The original OASIS notebook already contains the representation benchmark. This cell loads the saved result when available and refuses to call it complete otherwise.

In [7]:
fs_file=PAPER_ROOT/"fs_benchmark.csv"
if fs_file.exists():
    fs_benchmark=pd.read_csv(fs_file)
    print("Saved FreeSurfer benchmark found.")
    display(fs_benchmark)
    display(fs_benchmark.groupby("representation").agg(
        N_repeats=("repeat","count"),
        rho_mean=("disagreement_error_rho","mean"),
        rho_median=("disagreement_error_rho","median"),
        MAE_mean=("prediction_MAE","mean"),
        R2_mean=("prediction_R2","mean")
    ).reset_index())
    if fs_benchmark["repeat"].nunique()<3:
        print("WARNING: fewer than 3 FreeSurfer repeats are present.")
else:
    print("No fs_benchmark.csv found.")
    print("STATUS: FreeSurfer benchmark remains incomplete.")


Saved FreeSurfer benchmark found.


,repeat,representation,test_subjects,prediction_MAE,prediction_RMSE,prediction_R2,disagreement_error_rho,disagreement_error_p
0,0,Triad_rawT1,43,2.697558,5.393375,0.180680,0.456660,0.002083
1,0,FreeSurfer,43,2.538253,5.024043,0.289050,0.262609,0.088885
2,1,Triad_rawT1,43,1.525010,3.038840,0.076943,0.556478,0.000107
3,1,FreeSurfer,43,1.592759,2.771331,0.232304,0.597554,0.000023
4,2,Triad_rawT1,43,1.866296,3.138416,0.220021,0.498490,0.000669
5,2,FreeSurfer,43,1.598916,2.886483,0.340219,0.078677,0.616022


,representation,N_repeats,rho_mean,rho_median,MAE_mean,R2_mean
0,FreeSurfer,3,0.312947,0.262609,1.909976,0.287191
1,Triad_rawT1,3,0.503876,0.498490,2.029621,0.159215


## 7. Save all robustness outputs

In [8]:
save_map={
    "negative_control_shuffled_pairing.csv":negative_control,
    "horizon_stratified_reliability.csv":horizon_df,
    "history_difficulty_sensitivity.csv":sensitivity_df,
    "prediction_level_corruption_sensitivity.csv":corruption_df,
}
if "transfer_df" in globals():
    save_map["locked_oasis_to_adni_transfer.csv"]=transfer_df
if "fs_benchmark" in globals():
    save_map["freesurfer_benchmark_checkpoint.csv"]=fs_benchmark
for name,df in save_map.items():
    df.to_csv(PAPER_ROOT/name,index=False)
print("Saved",len(save_map),"tables to",PAPER_ROOT)


Saved 5 tables to C:\Projects\OASIS\oasis_raw_t1_encoder\PAPER_READY_FINAL_RESULTS


## Interpretation guardrails
Negative controls, horizon stratification, history sensitivity, and prediction-level corruption are robustness analyses. The external-transfer table is the important cross-dataset validation. None of these analyses should be used to manufacture significance. Report effect sizes, uncertainty intervals, and event/sample counts.